# 02b — Deduplicación Cross-Portal

**Pipeline:** `00_Setup_Mount` → `02_Silver_Layer` → **`02b_Dedup_Cross_Portal`** → `03_Gold_Layer` → `04_Model_Training`

**Problema:** El mismo apartamento en Chapinero puede estar en FincaRaíz a $350M, CienCuadras a $345M y Metrocuadrado a $360M. Sin dedup cross-portal, el modelo se entrena con 3 filas del mismo inmueble con 3 precios distintos → ruido sistemático.

**Solución:**
1. Normalización de ubicación (acentos, case, separadores).
2. Blocking por `(ciudad, habitaciones)` para evitar O(n²).
3. Matching por Jaccard de ubicación + similitud de área + ratio de precio.
4. Componentes conectados para agrupar matches transitivos.
5. Selección de representante para ML (mejor completitud, precio mediano del grupo).
6. **Inteligencia de precios cross-portal** → ventaja competitiva.

**Output:**
- `silver/master_deduped/` — 1 fila por inmueble real (para Gold + ML).
- `gold/price_intelligence/` — comparación de precios cross-portal.

In [ ]:
# =============================================================
# CELDA 1: SETUP — Imports, Credenciales, Config S3
# =============================================================
import json
import sys
import os
import importlib

from pyspark.sql import functions as F

# ── Resolver path del módulo dedup_cross_portal.py ──
# Databricks Repos: los .py del repo se importan directo.
# VS Code local: el CWD del kernel puede no ser el directorio del notebook.
_notebook_dir = os.path.dirname(os.path.abspath(
    globals().get("__vsc_ipynb_file__", "")
)) if globals().get("__vsc_ipynb_file__") else None

for _candidate in filter(None, [_notebook_dir, os.getcwd()]):
    if _candidate not in sys.path:
        sys.path.insert(0, _candidate)

# Credenciales (misma lógica que 02_Silver_Layer)
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
except FileNotFoundError:
    raise SystemExit("❌ aws_secrets.json no encontrado.")
except json.JSONDecodeError:
    raise SystemExit("❌ aws_secrets.json inválido.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

# Importar módulo de dedup (forzar recarga si se edita el .py entre ejecuciones)
import dedup_cross_portal
importlib.reload(dedup_cross_portal)

from dedup_cross_portal import (
    run_cross_portal_dedup,
    prepare_for_matching,
    find_cross_portal_matches,
    assign_property_groups,
    select_ml_representative,
    build_price_intelligence,
)

print(f"✅ Setup Dedup Cross-Portal listo.")
print(f"   Módulo cargado desde: {dedup_cross_portal.__file__}")

In [ ]:
# =============================================================
# CELDA 2: LEER SILVER — Delta desde S3
# =============================================================

ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
reader = spark.read.format("delta")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_silver = reader.load(ruta_silver)

total = df_silver.count()
portales = df_silver.select("fuente").distinct().count()
print(f"📖 Silver: {total:,} registros de {portales} portales")
print(f"   Columnas: {df_silver.columns}")

# Vista rápida por portal
df_silver.groupBy("fuente").count().orderBy(F.desc("count")).show(truncate=False)

In [ ]:
# =============================================================
# CELDA 3: EJECUTAR DEDUP CROSS-PORTAL
# =============================================================
# Parámetros configurables (ajustar según resultados de auditoría)
#   area_tolerance:          ±10% de diferencia de área para considerar match
#   location_sim_threshold:  50% de tokens de ubicación en común (Jaccard)
#   max_price_ratio:         Precio máximo 1.25x entre portales (±25%)

results = run_cross_portal_dedup(
    df_silver,
    area_tolerance=0.10,           # ±10% de área
    location_sim_threshold=0.50,   # ≥50% tokens ubicación en común
    max_price_ratio=1.25,          # Precio portal A máx 1.25x portal B
)

df_ml_clean = results["df_ml_clean"]
df_intelligence = results["df_intelligence"]
df_price_detail = results["df_price_detail"]
df_pairs = results["df_pairs"]
stats = results["stats"]

print(f"\n📊 Resumen:")
for k, v in stats.items():
    print(f"   {k}: {v}")

In [ ]:
# =============================================================
# CELDA 4: AUDITORÍA — Revisar pares matched (spot-check)
# =============================================================
# Revisar los matches de mayor confianza para validar calidad

print("🔍 Top 20 pares matched (mayor score):")
display(
    df_pairs
    .orderBy(F.desc("match_score"))
    .select(
        "fuente_a", "ubicacion_a", "precio_a", "area_a",
        "fuente_b", "ubicacion_b", "precio_b", "area_b",
        "jaccard_sim", "area_sim", "price_sim", "match_score",
    )
    .limit(20)
)

# Distribución de scores para calibrar umbral
print("\n📈 Distribución de match_score:")
df_pairs.select(
    F.count("*").alias("total_pares"),
    F.round(F.avg("match_score"), 3).alias("score_promedio"),
    F.round(F.min("match_score"), 3).alias("score_min"),
    F.round(F.max("match_score"), 3).alias("score_max"),
    F.round(F.expr("percentile_approx(match_score, 0.25)"), 3).alias("p25"),
    F.round(F.expr("percentile_approx(match_score, 0.50)"), 3).alias("p50"),
    F.round(F.expr("percentile_approx(match_score, 0.75)"), 3).alias("p75"),
).show(truncate=False)

In [ ]:
# =============================================================
# CELDA 5: INTELIGENCIA DE PRECIOS — Ventaja Competitiva
# =============================================================

if stats["price_intelligence_count"] > 0:
    print("💰 INTELIGENCIA DE NEGOCIACIÓN — Inmuebles en múltiples portales\n")

    # Top oportunidades de ahorro
    print("🏆 Top 15 oportunidades de ahorro:")
    display(
        df_intelligence
        .select(
            "titulo_inmueble",
            "ubicacion_raw",
            "area_m2",
            "habitaciones",
            "num_portales",
            "portal_mas_barato",
            F.format_number("precio_portal_barato", 0).alias("precio_barato"),
            "portal_mas_caro",
            F.format_number("precio_portal_caro", 0).alias("precio_caro"),
            F.format_number("ahorro_potencial", 0).alias("ahorro_COP"),
            F.col("ahorro_pct"),
        )
        .orderBy(F.desc("ahorro_potencial"))
        .limit(15)
    )

    # Resumen por portal: ¿quién tiende a ser más caro?
    print("\n📊 ¿Qué portal tiende a ser más caro/barato?")
    display(
        df_intelligence
        .groupBy("portal_mas_barato")
        .agg(
            F.count("*").alias("veces_mas_barato"),
            F.round(F.avg("ahorro_potencial"), 0).alias("ahorro_promedio"),
        )
        .orderBy(F.desc("veces_mas_barato"))
    )

    display(
        df_intelligence
        .groupBy("portal_mas_caro")
        .agg(
            F.count("*").alias("veces_mas_caro"),
            F.round(F.avg("ahorro_potencial"), 0).alias("sobrecosto_promedio"),
        )
        .orderBy(F.desc("veces_mas_caro"))
    )

    # Detalle: precios por portal para cada inmueble duplicado
    print("\n🔎 Detalle de precios por portal (drill-down):")
    display(df_price_detail.limit(30))
else:
    print("ℹ️ No se encontraron inmuebles en múltiples portales en este batch.")

In [ ]:
# =============================================================
# CELDA 6: GUARDAR — Silver Deduped + Gold Price Intelligence
# =============================================================

# ── Silver Deduped: tabla limpia para Gold + ML ──
ruta_deduped = f"s3a://{BUCKET}/silver/master_deduped/"

# Seleccionar columnas Silver estándar + columnas nuevas de dedup
cols_silver = [
    "id_original", "fecha_extraccion", "fuente", "url", "titulo",
    "ubicacion_raw", "precio_num", "area_m2", "habitaciones", "banos",
    "garajes", "tipo_inmueble", "estado_inmueble", "source_file", "batch_id",
    "silver_processed_at",
    # Columnas nuevas de dedup
    "property_group_id", "num_portales", "precio_mediano_grupo",
    "precio_min_grupo", "precio_max_grupo", "precio_original_portal",
    "data_completeness",
]

# Solo guardar columnas que existen (flexibilidad)
existing_cols = set(df_ml_clean.columns)
cols_to_save = [c for c in cols_silver if c in existing_cols]

writer = (
    df_ml_clean
    .select(*cols_to_save)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
)
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_deduped)
print(f"🥈 Silver deduped: {ruta_deduped}")
print(f"   {df_ml_clean.count():,} registros (1 por inmueble real)")

# ── Gold consumable deduped: snapshot para API ──
ruta_consumable = f"s3a://{BUCKET}/gold/app_consumable/"
writer = df_ml_clean.select(*cols_to_save).coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_consumable)
print(f"🥇 Gold consumable (deduped): {ruta_consumable}")

# ── Gold Price Intelligence ──
if stats["price_intelligence_count"] > 0:
    ruta_intel = f"s3a://{BUCKET}/gold/price_intelligence/"
    writer = (
        df_intelligence
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    for k, v in S3_OPTS.items():
        writer = writer.option(k, v)
    writer.save(ruta_intel)
    print(f"💰 Price intelligence: {ruta_intel}")
    print(f"   {stats['price_intelligence_count']:,} inmuebles con precio en múltiples portales")

    # Detalle por portal (para drill-down en app)
    ruta_detail = f"s3a://{BUCKET}/gold/price_detail/"
    writer = df_price_detail.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    for k, v in S3_OPTS.items():
        writer = writer.option(k, v)
    writer.save(ruta_detail)
    print(f"📋 Price detail: {ruta_detail}")

print("\n✅ Dedup Cross-Portal completo. Pipeline listo para 03_Gold_Layer.")
